# 05. ALS шаг за шагом

Сначала подробно разбираем validation: interactions, confidence, mappings, CSR, fit и recommend. Train и test повторяют уже понятные шаги без цикла по окнам.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Mounted at /content/drive
Корень проекта: /content/drive/MyDrive/fashion-recommender-system


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [2]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем sparse matrix, implicit ALS и технический массовый inference  
**Зачем:** основная подготовка validation всё равно выполняется напрямую  
**Что получим:** необходимые классы и метрики

In [3]:
import time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from IPython.display import display

from fashion_recommender.als import (
    InteractionMatrix, generate_als_candidates, prepare_user_item_matrix,
)
from fashion_recommender.data import load_transactions
from fashion_recommender.evaluation import (
    candidate_recall_at_k, hit_rate_at_k, map_at_k, mean_recall_at_k,
)
from fashion_recommender.persistence import load_json, save_als_model, save_json

/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


### Пути и ALS-параметры

**Что делаем:** проверяем temporal windows и задаём один набор гиперпараметров  
**Зачем:** никаких скрытых настроек между окнами быть не должно  
**Что получим:** пути и константы

In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

WINDOWS_PATH = PROCESSED_DIR / "temporal_windows.json"
if not WINDOWS_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден файл: {WINDOWS_PATH}\n"
        "Сначала выполните notebook 03_temporal_validation_colab.ipynb."
    )

FACTORS = 64
REGULARIZATION = 0.05
ITERATIONS = 20
CANDIDATE_LIMIT = 200
MAX_EVALUATION_USERS = 2_000
RANDOM_STATE = 42

### Загрузка входов

**Что делаем:** читаем transactions и JSON окон  
**Зачем:** данные загружаются один раз  
**Что получим:** `transactions` и `windows`

In [5]:
transactions = load_transactions(TRANSACTIONS_PATH)
windows = load_json(WINDOWS_PATH)
print("Transactions:", transactions.shape)

Transactions: (1048575, 5)


### Validation-границы

**Что делаем:** выбираем одно окно для подробного разбора  
**Зачем:** validation не является final test  
**Что получим:** `validation_cutoff` и `validation_end`

In [6]:
validation_cutoff = pd.Timestamp(windows["validation"]["cutoff_date"])
validation_end = pd.Timestamp(windows["validation"]["target_end_date"])
print("Validation:", validation_cutoff.date(), "—", validation_end.date())

Validation: 2019-12-18 — 2019-12-24


### Validation history

**Что делаем:** берём покупки строго раньше cutoff  
**Зачем:** ALS matrix не должна видеть validation target  
**Что получим:** `validation_history`

In [7]:
validation_history = transactions[
    transactions["t_dat"] < validation_cutoff
].copy()
print("Validation history:", validation_history.shape)
assert validation_history["t_dat"].max() < validation_cutoff

Validation history: (1011880, 5)


### Validation target

**Что делаем:** берём только семь дней validation  
**Зачем:** эта таблица используется для ground truth, не для fit  
**Что получим:** `validation_target`

In [8]:
validation_target = transactions[
    transactions["t_dat"].between(validation_cutoff, validation_end)
].copy()
print("Validation target:", validation_target.shape)

Validation target: (23405, 5)


### Известные validation users

**Что делаем:** фильтруем cold-start клиентов  
**Зачем:** ALS не имеет factor для пользователя без history  
**Что получим:** `validation_target_evaluation`

In [9]:
validation_known_users = set(validation_history["customer_id"])
validation_target_evaluation = validation_target[
    validation_target["customer_id"].isin(validation_known_users)
].copy()
print("Known target users:", validation_target_evaluation["customer_id"].nunique())

Known target users: 13078


### Уникальные validation pairs

**Что делаем:** сортируем future и удаляем повторные пары  
**Зачем:** ground truth учитывает товар один раз  
**Что получим:** `validation_target_unique`

In [10]:
validation_target_unique = (
    validation_target_evaluation
    .sort_values("t_dat")
    .drop_duplicates(["customer_id", "article_id"])
)
display(validation_target_unique.head())

,t_dat,customer_id,article_id,price,sales_channel_id
519889,2019-12-18,b40319c58fa8fece06520aa53d70a14646da8b75972e9d...,0756099005,0.016932,2
215618,2019-12-18,7cc44f8f11562456b838fa66fd66e3d326b9ad8999316e...,0813898002,0.025407,2
215696,2019-12-18,0d744e9c959f3ab3d5c274cf8c315bb15a27443fadb50e...,0781613010,0.020576,2
862161,2019-12-18,4062b13a40e2b5ab05a63c876e823a5d496535efa73dfc...,0874240002,0.030492,2
644878,2019-12-18,107fc61c56d5ce54881fe88c436bfc0cf3887ba0ef5003...,0740812007,0.013712,2


### Validation ground truth

**Что делаем:** собираем будущие товары каждого пользователя  
**Зачем:** словарь пока остаётся полным  
**Что получим:** `validation_ground_truth`

In [11]:
validation_ground_truth = (
    validation_target_unique
    .groupby("customer_id", sort=False)["article_id"]
    .apply(list)
    .to_dict()
)
print("Ground-truth users:", len(validation_ground_truth))

Ground-truth users: 13078


### Validation cohort

**Что делаем:** случайно выбираем до 2 000 ID с фиксированным seed  
**Зачем:** порядок словаря не влияет на эксперимент  
**Что получим:** отдельный `validation_ground_truth_sample`

In [12]:
validation_all_users = np.array(sorted(validation_ground_truth))
validation_sample_size = min(MAX_EVALUATION_USERS, len(validation_all_users))
validation_rng = np.random.default_rng(RANDOM_STATE)
validation_users = validation_rng.choice(
    validation_all_users,
    size=validation_sample_size,
    replace=False,
).tolist()
validation_ground_truth_sample = {
    customer_id: validation_ground_truth[customer_id]
    for customer_id in validation_users
}
print("Evaluation users:", len(validation_users))

Evaluation users: 2000


### User-item counts

**Что делаем:** агрегируем повторные покупки пары  
**Зачем:** implicit ALS работает с силой наблюдаемого сигнала  
**Что получим:** `validation_interaction_counts`

In [13]:
validation_interaction_counts = (
    validation_history
    .groupby(["customer_id", "article_id"])
    .size()
    .reset_index(name="purchase_count")
)
display(validation_interaction_counts.head())

,customer_id,article_id,purchase_count
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601006,1
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0797065001,1
2,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0611584007,1
3,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0640021005,1
4,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0689898002,1


### Confidence

**Что делаем:** преобразуем purchase count логарифмом  
**Зачем:** повторы усиливают сигнал без линейного роста  
**Что получим:** столбец `confidence`

In [14]:
validation_interaction_counts["confidence"] = (
    1 + np.log1p(validation_interaction_counts["purchase_count"])
)
display(validation_interaction_counts.head())

,customer_id,article_id,purchase_count,confidence
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601006,1,1.693147
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0797065001,1,1.693147
2,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0611584007,1,1.693147
3,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0640021005,1,1.693147
4,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0689898002,1,1.693147


### Списки ID

**Что делаем:** создаём детерминированные списки users и items  
**Зачем:** номер строки/столбца должен иметь обратное отображение  
**Что получим:** `validation_index_to_user` и `validation_index_to_item`

In [15]:
validation_index_to_user = sorted(
    validation_interaction_counts["customer_id"].astype(str).unique()
)
validation_index_to_item = sorted(
    validation_interaction_counts["article_id"].astype(str).unique()
)
print("Users:", len(validation_index_to_user))
print("Items:", len(validation_index_to_item))

Users: 447850
Items: 50291


### Mappings

**Что делаем:** назначаем числовой индекс каждому исходному ID  
**Зачем:** sparse matrix принимает только числовые координаты  
**Что получим:** два словаря ID → index

In [16]:
validation_user_to_index = {
    customer_id: index
    for index, customer_id in enumerate(validation_index_to_user)
}
validation_item_to_index = {
    article_id: index
    for index, article_id in enumerate(validation_index_to_item)
}
print("User mapping example:", next(iter(validation_user_to_index.items())))
print("Item mapping example:", next(iter(validation_item_to_index.items())))

User mapping example: ('00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657', 0)
Item mapping example: ('0108775015', 0)


### Числовые индексы

**Что делаем:** применяем mappings к таблице взаимодействий  
**Зачем:** каждая пара получает координату CSR  
**Что получим:** `user_index` и `item_index`

In [17]:
validation_interaction_counts["user_index"] = (
    validation_interaction_counts["customer_id"].astype(str).map(validation_user_to_index)
)
validation_interaction_counts["item_index"] = (
    validation_interaction_counts["article_id"].astype(str).map(validation_item_to_index)
)
display(validation_interaction_counts.head())

,customer_id,article_id,purchase_count,confidence,user_index,item_index
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601006,1,1.693147,0,6932
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0797065001,1,1.693147,0,46534
2,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0611584007,1,1.693147,1,10810
3,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0640021005,1,1.693147,1,15331
4,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0689898002,1,1.693147,1,23866


### CSR matrix

**Что делаем:** передаём confidence и координаты в `csr_matrix`  
**Зачем:** нули не хранятся в памяти  
**Что получим:** `validation_user_item_matrix`

In [18]:
validation_user_item_matrix = csr_matrix(
    (
        validation_interaction_counts["confidence"].astype("float32"),
        (
            validation_interaction_counts["user_index"],
            validation_interaction_counts["item_index"],
        ),
    ),
    shape=(len(validation_index_to_user), len(validation_index_to_item)),
    dtype=np.float32,
)

### Размер sparse matrix

**Что делаем:** смотрим shape, nnz и density  
**Зачем:** density должна быть очень маленькой  
**Что получим:** проверку памяти и ориентации users × items

In [19]:
validation_density = (
    validation_user_item_matrix.nnz
    / (validation_user_item_matrix.shape[0] * validation_user_item_matrix.shape[1])
)
print("Type:", type(validation_user_item_matrix))
print("Shape:", validation_user_item_matrix.shape)
print("NNZ:", validation_user_item_matrix.nnz)
print("Density:", validation_density)

Type: <class 'scipy.sparse._csr.csr_matrix'>
Shape: (447850, 50291)
NNZ: 998998
Density: 4.435491679355036e-05


### Создание ALS

**Что делаем:** задаём модель до обучения  
**Зачем:** создание объекта и fit — разные действия  
**Что получим:** `validation_model`

In [20]:
validation_model = AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=RANDOM_STATE,
)
print(validation_model)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


### Обучение ALS

**Что делаем:** вызываем `.fit()` на user-item CSR  
**Зачем:** target не передаётся модели  
**Что получим:** обученные user/item factors

In [21]:
validation_fit_started = time.perf_counter()
validation_model.fit(validation_user_item_matrix, show_progress=False)
validation_training_time = time.perf_counter() - validation_fit_started
print("User factors:", validation_model.user_factors.shape)
print("Item factors:", validation_model.item_factors.shape)
print("Training seconds:", round(validation_training_time, 2))

User factors: (447850, 64)
Item factors: (50291, 64)
Training seconds: 53.34


### Один пользователь

**Что делаем:** находим matrix index первого evaluation user  
**Зачем:** сначала проверяем API recommend на одном примере  
**Что получим:** `validation_example_user_index`

In [22]:
validation_example_user = validation_users[0]
validation_example_user_index = validation_user_to_index[validation_example_user]
print("Customer:", validation_example_user)
print("Matrix row:", validation_example_user_index)

Customer: f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773caf840ca3abcd7fc9de
Matrix row: 426343


### Один вызов recommend

**Что делаем:** получаем item indices и scores  
**Зачем:** проверяем фактический API установленного implicit  
**Что получим:** два NumPy-массива

In [23]:
validation_example_indices, validation_example_scores = validation_model.recommend(
    validation_example_user_index,
    validation_user_item_matrix[validation_example_user_index],
    N=10,
    filter_already_liked_items=True,
)
print("Indices:", validation_example_indices[:5])
print("Scores:", validation_example_scores[:5])

Indices: [   45   924  1236 23750 31775]
Scores: [0.06961982 0.06445309 0.04293447 0.04125626 0.04000032]


### Обратное преобразование ID

**Что делаем:** заменяем item indices исходными article ID  
**Зачем:** API и метрики работают с бизнес-идентификаторами  
**Что получим:** таблицу десяти рекомендаций

In [24]:
validation_example_articles = [
    validation_index_to_item[int(item_index)]
    for item_index in validation_example_indices
]
validation_example_table = pd.DataFrame({
    "article_id": validation_example_articles,
    "als_score": validation_example_scores,
})
display(validation_example_table)

,article_id,als_score
0,0156231001,0.069620
1,0372860002,0.064453
2,0399256005,0.042934
3,0689109001,0.041256
4,0720125001,0.040000
5,0158340001,0.038926
6,0699075001,0.034768
7,0712587003,0.033958
8,0656763001,0.030546
9,0399223033,0.027656


### Технический контейнер

**Что делаем:** упаковываем matrix и mappings для массового recommend  
**Зачем:** эта операция не скрывает уже показанную подготовку  
**Что получим:** `validation_interactions`

In [25]:
validation_interactions = InteractionMatrix(
    matrix=validation_user_item_matrix,
    user_to_index=validation_user_to_index,
    item_to_index=validation_item_to_index,
    index_to_user=validation_index_to_user,
    index_to_item=validation_index_to_item,
)

### Массовые validation candidates

**Что делаем:** вызываем переиспользуемый batched recommend  
**Зачем:** цикл по пользователям является техническим inference, не циклом по окнам  
**Что получим:** `validation_candidates`

In [26]:
validation_candidates = generate_als_candidates(
    validation_model,
    validation_interactions,
    customer_ids=validation_users,
    limit=CANDIDATE_LIMIT,
)
print("Candidate rows:", len(validation_candidates))
display(validation_candidates.head())

Candidate rows: 400000


,customer_id,article_id,als_score,als_rank
0,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0156231001,0.069620,1
1,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0372860002,0.064453,2
2,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0399256005,0.042934,3
3,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0689109001,0.041256,4
4,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0720125001,0.040000,5


### Validation Candidate Recall

**Что делаем:** собираем candidate lists и оцениваем покрытие  
**Зачем:** ранжировщик не найдёт товар вне candidates  
**Что получим:** Candidate Recall@CANDIDATE_LIMIT

In [27]:
validation_candidate_lists = validation_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
validation_candidate_recall = candidate_recall_at_k(
    validation_ground_truth_sample,
    validation_candidate_lists,
    CANDIDATE_LIMIT,
)
print(f"Candidate Recall@{CANDIDATE_LIMIT}:", validation_candidate_recall)

Candidate Recall@200: 0.04066666666666666


### Validation Top-12

**Что делаем:** оцениваем первые 12 ALS items  
**Зачем:** standalone качество сравнимо с baseline  
**Что получим:** Recall/MAP/HitRate

In [28]:
validation_top12_lists = validation_candidates[
    validation_candidates["als_rank"] <= 12
].groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()

validation_metrics = {
    "Recall@12": mean_recall_at_k(validation_ground_truth_sample, validation_top12_lists, 12),
    "MAP@12": map_at_k(validation_ground_truth_sample, validation_top12_lists, 12),
    "HitRate@12": hit_rate_at_k(validation_ground_truth_sample, validation_top12_lists, 12),
}
display(pd.Series(validation_metrics))

,0
Recall@12,0.003250
MAP@12,0.001187
HitRate@12,0.003500


### Сохранение validation candidates

**Что делаем:** записываем candidate table отдельно от модели  
**Зачем:** notebook 07 сможет загрузить её напрямую  
**Что получим:** `als_candidates_validation.parquet`

In [29]:
validation_candidates_path = PROCESSED_DIR / "als_candidates_validation.parquet"
validation_candidates.to_parquet(validation_candidates_path, index=False)
print("Сохранено:", validation_candidates_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/als_candidates_validation.parquet


### Ground truth helper

**Что делаем:** фиксируем уже показанные pandas-шаги для других окон  
**Зачем:** функция не обучает и ничего не сохраняет  
**Что получим:** небольшую `build_ground_truth`

In [30]:
def build_ground_truth(target, known_users):
    target_evaluation = target[target["customer_id"].isin(known_users)]
    target_unique = (
        target_evaluation
        .sort_values("t_dat")
        .drop_duplicates(["customer_id", "article_id"])
    )
    return (
        target_unique
        .groupby("customer_id", sort=False)["article_id"]
        .apply(list)
        .to_dict()
    )

### Sampling helper

**Что делаем:** фиксируем воспроизводимый выбор cohort  
**Зачем:** полный ground truth не перезаписывается  
**Что получим:** users и отдельный sample dictionary

In [31]:
def sample_ground_truth(ground_truth, max_users, random_state):
    all_users = np.array(sorted(ground_truth))
    sample_size = min(max_users, len(all_users))
    rng = np.random.default_rng(random_state)
    users = rng.choice(all_users, size=sample_size, replace=False).tolist()
    ground_truth_sample = {
        customer_id: ground_truth[customer_id]
        for customer_id in users
    }
    return users, ground_truth_sample

### Train: границы

**Что делаем:** выбираем train-окно  
**Зачем:** validation уже показал устройство ALS подробно  
**Что получим:** `train_cutoff` и `train_end`

In [32]:
train_cutoff = pd.Timestamp(windows["train"]["cutoff_date"])
train_end = pd.Timestamp(windows["train"]["target_end_date"])
print("Train:", train_cutoff.date(), "—", train_end.date())

Train: 2019-12-11 — 2019-12-17


### Train: history и target

**Что делаем:** делим транзакции по выбранным датам  
**Зачем:** ALS должен видеть только строки до cutoff  
**Что получим:** `train_history` и `train_target`

In [33]:
train_history = transactions[
    transactions["t_dat"] < train_cutoff
].copy()
train_target = transactions[
    transactions["t_dat"].between(train_cutoff, train_end)
].copy()

assert train_history["t_dat"].max() < train_cutoff
print("History:", train_history.shape, "Target:", train_target.shape)

History: (998087, 5) Target: (13793, 5)


### Train: ground truth

**Что делаем:** повторяем уже разобранную подготовку ответов  
**Зачем:** evaluation использует полный future список  
**Что получим:** `train_ground_truth`

In [34]:
train_ground_truth = build_ground_truth(
    train_target,
    set(train_history["customer_id"]),
)
print("Ground-truth users:", len(train_ground_truth))

Ground-truth users: 7812


### Train: cohort

**Что делаем:** делаем воспроизводимую выборку пользователей  
**Зачем:** ALS и Content-Based должны работать на одинаковом cohort  
**Что получим:** `train_users` и `train_ground_truth_sample`

In [35]:
train_users, train_ground_truth_sample = sample_ground_truth(
    train_ground_truth,
    MAX_EVALUATION_USERS,
    RANDOM_STATE,
)
print("Evaluation users:", len(train_users))

Evaluation users: 2000


### Train: sparse inputs

**Что делаем:** повторяем показанные groupby, mappings и CSR технической функцией  
**Зачем:** подробная реализация уже была видна на validation  
**Что получим:** `train_interactions`

In [36]:
train_interactions = prepare_user_item_matrix(train_history)
print("Matrix:", train_interactions.matrix.shape)
print("NNZ:", train_interactions.matrix.nnz)

Matrix: (443629, 49712)
NNZ: 985415


### Train: модель

**Что делаем:** создаём отдельный ALS для этого cutoff  
**Зачем:** разные окна не должны использовать future interactions  
**Что получим:** `train_model` до обучения

In [37]:
train_model = AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=RANDOM_STATE,
)
print(train_model)

### Train: обучение

**Что делаем:** вызываем `.fit()` только на history matrix  
**Зачем:** оценка и сохранение остаются в следующих ячейках  
**Что получим:** обученный `train_model`

In [38]:
train_fit_started = time.perf_counter()
train_model.fit(train_interactions.matrix, show_progress=False)
train_training_time = time.perf_counter() - train_fit_started
print("Training seconds:", round(train_training_time, 2))

Training seconds: 55.67


### Train: массовые кандидаты

**Что делаем:** получаем Top-K для evaluation cohort  
**Зачем:** массовый inference является разрешённой технической функцией  
**Что получим:** `train_candidates`

In [39]:
train_candidates = generate_als_candidates(
    train_model,
    train_interactions,
    customer_ids=train_users,
    limit=CANDIDATE_LIMIT,
)
print("Candidate rows:", len(train_candidates))
display(train_candidates.head())

Candidate rows: 400000


,customer_id,article_id,als_score,als_rank
0,565e138a74f621194fba850fd52c43c2fd462714437748...,0554479005,0.028321,1
1,565e138a74f621194fba850fd52c43c2fd462714437748...,0719655001,0.026284,2
2,565e138a74f621194fba850fd52c43c2fd462714437748...,0448509014,0.024535,3
3,565e138a74f621194fba850fd52c43c2fd462714437748...,0656763001,0.020662,4
4,565e138a74f621194fba850fd52c43c2fd462714437748...,0678942001,0.020532,5


### Train: метрики

**Что делаем:** считаем Candidate Recall и standalone Top-12  
**Зачем:** оценка не смешана с fit или сохранением  
**Что получим:** `train_metrics`

In [40]:
train_candidate_lists = train_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
train_top12_lists = train_candidates[
    train_candidates["als_rank"] <= 12
].groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()
train_metrics = {
    "Candidate Recall": candidate_recall_at_k(
        train_ground_truth_sample, train_candidate_lists, CANDIDATE_LIMIT
    ),
    "Recall@12": mean_recall_at_k(train_ground_truth_sample, train_top12_lists, 12),
    "MAP@12": map_at_k(train_ground_truth_sample, train_top12_lists, 12),
    "HitRate@12": hit_rate_at_k(train_ground_truth_sample, train_top12_lists, 12),
}
display(pd.Series(train_metrics))

,0
Candidate Recall,0.047583
Recall@12,0.004250
MAP@12,0.001095
HitRate@12,0.005000


### Train: сохранение кандидатов

**Что делаем:** записываем только candidate table  
**Зачем:** notebook 07 загрузит файл без повторного ALS fit  
**Что получим:** `als_candidates_train.parquet`

In [41]:
train_candidates_path = PROCESSED_DIR / "als_candidates_train.parquet"
train_candidates.to_parquet(train_candidates_path, index=False)
print("Сохранено:", train_candidates_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/als_candidates_train.parquet


### Test: границы

**Что делаем:** выбираем test-окно  
**Зачем:** validation уже показал устройство ALS подробно  
**Что получим:** `test_cutoff` и `test_end`

In [42]:
test_cutoff = pd.Timestamp(windows["test"]["cutoff_date"])
test_end = pd.Timestamp(windows["test"]["target_end_date"])
print("Test:", test_cutoff.date(), "—", test_end.date())

Test: 2019-12-25 — 2019-12-31


### Test: history и target

**Что делаем:** делим транзакции по выбранным датам  
**Зачем:** ALS должен видеть только строки до cutoff  
**Что получим:** `test_history` и `test_target`

In [43]:
test_history = transactions[
    transactions["t_dat"] < test_cutoff
].copy()
test_target = transactions[
    transactions["t_dat"].between(test_cutoff, test_end)
].copy()

assert test_history["t_dat"].max() < test_cutoff
print("History:", test_history.shape, "Target:", test_target.shape)

History: (1035285, 5) Target: (13290, 5)


### Test: ground truth

**Что делаем:** повторяем уже разобранную подготовку ответов  
**Зачем:** evaluation использует полный future список  
**Что получим:** `test_ground_truth`

In [44]:
test_ground_truth = build_ground_truth(
    test_target,
    set(test_history["customer_id"]),
)
print("Ground-truth users:", len(test_ground_truth))

Ground-truth users: 7590


### Test: cohort

**Что делаем:** делаем воспроизводимую выборку пользователей  
**Зачем:** ALS и Content-Based должны работать на одинаковом cohort  
**Что получим:** `test_users` и `test_ground_truth_sample`

In [45]:
test_users, test_ground_truth_sample = sample_ground_truth(
    test_ground_truth,
    MAX_EVALUATION_USERS,
    RANDOM_STATE,
)
print("Evaluation users:", len(test_users))

Evaluation users: 2000


### Test: sparse inputs

**Что делаем:** повторяем показанные groupby, mappings и CSR технической функцией  
**Зачем:** подробная реализация уже была видна на validation  
**Что получим:** `test_interactions`

In [46]:
test_interactions = prepare_user_item_matrix(test_history)
print("Matrix:", test_interactions.matrix.shape)
print("NNZ:", test_interactions.matrix.nnz)

Matrix: (454441, 50928)
NNZ: 1022169


### Test: модель

**Что делаем:** создаём отдельный ALS для этого cutoff  
**Зачем:** разные окна не должны использовать future interactions  
**Что получим:** `test_model` до обучения

In [47]:
test_model = AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=RANDOM_STATE,
)
print(test_model)

### Test: обучение

**Что делаем:** вызываем `.fit()` только на history matrix  
**Зачем:** оценка и сохранение остаются в следующих ячейках  
**Что получим:** обученный `test_model`

In [48]:
test_fit_started = time.perf_counter()
test_model.fit(test_interactions.matrix, show_progress=False)
test_training_time = time.perf_counter() - test_fit_started
print("Training seconds:", round(test_training_time, 2))

Training seconds: 53.61


### Test: массовые кандидаты

**Что делаем:** получаем Top-K для evaluation cohort  
**Зачем:** массовый inference является разрешённой технической функцией  
**Что получим:** `test_candidates`

In [49]:
test_candidates = generate_als_candidates(
    test_model,
    test_interactions,
    customer_ids=test_users,
    limit=CANDIDATE_LIMIT,
)
print("Candidate rows:", len(test_candidates))
display(test_candidates.head())

Candidate rows: 400000


,customer_id,article_id,als_score,als_rank
0,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0739144004,0.022007,1
1,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0788261001,0.021915,2
2,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0554450001,0.018854,3
3,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0748269003,0.017985,4
4,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0484398001,0.017925,5


### Test: метрики

**Что делаем:** считаем Candidate Recall и standalone Top-12  
**Зачем:** оценка не смешана с fit или сохранением  
**Что получим:** `test_metrics`

In [50]:
test_candidate_lists = test_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
test_top12_lists = test_candidates[
    test_candidates["als_rank"] <= 12
].groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()
test_metrics = {
    "Candidate Recall": candidate_recall_at_k(
        test_ground_truth_sample, test_candidate_lists, CANDIDATE_LIMIT
    ),
    "Recall@12": mean_recall_at_k(test_ground_truth_sample, test_top12_lists, 12),
    "MAP@12": map_at_k(test_ground_truth_sample, test_top12_lists, 12),
    "HitRate@12": hit_rate_at_k(test_ground_truth_sample, test_top12_lists, 12),
}
display(pd.Series(test_metrics))

,0
Candidate Recall,0.040850
Recall@12,0.004767
MAP@12,0.001333
HitRate@12,0.005500


### Test: сохранение кандидатов

**Что делаем:** записываем только candidate table  
**Зачем:** notebook 07 загрузит файл без повторного ALS fit  
**Что получим:** `als_candidates_test.parquet`

In [51]:
test_candidates_path = PROCESSED_DIR / "als_candidates_test.parquet"
test_candidates.to_parquet(test_candidates_path, index=False)
print("Сохранено:", test_candidates_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/als_candidates_test.parquet


### Test alias

**Что делаем:** сохраняем удобное имя test candidates  
**Зачем:** старые consumers остаются совместимыми  
**Что получим:** `als_candidates.parquet`

In [52]:
als_alias_path = PROCESSED_DIR / "als_candidates.parquet"
test_candidates.to_parquet(als_alias_path, index=False)
print("Сохранено:", als_alias_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/als_candidates.parquet


### Финальные ALS interactions

**Что делаем:** строим matrix по всей доступной истории отдельно от offline test  
**Зачем:** API-модель не должна быть test-моделью, обученной только до cutoff  
**Что получим:** `final_interactions`

In [53]:
final_interactions = prepare_user_item_matrix(transactions)
print("Final matrix:", final_interactions.matrix.shape)
print("Final NNZ:", final_interactions.matrix.nnz)

Final matrix: (458235, 51279)
Final NNZ: 1035303


### Финальная ALS-модель

**Что делаем:** создаём новый объект для batch/API  
**Зачем:** offline metrics уже рассчитаны test-моделью  
**Что получим:** необученный `final_als_model`

In [54]:
final_als_model = AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=RANDOM_STATE,
)
print(final_als_model)

### Обучение финальной ALS

**Что делаем:** обучаем модель на всех доступных transactions  
**Зачем:** она сохраняется для дополнительного batch inference  
**Что получим:** обученный `final_als_model`

In [55]:
final_fit_started = time.perf_counter()
final_als_model.fit(final_interactions.matrix, show_progress=False)
final_training_time = time.perf_counter() - final_fit_started
print("Final training seconds:", round(final_training_time, 2))

Final training seconds: 53.14


### Сохранение ALS model

**Что делаем:** записываем модель отдельно от mappings  
**Зачем:** batch inference загрузит factor matrices  
**Что получим:** `models/als_model.npz`

In [56]:
als_model_path = save_als_model(
    final_als_model,
    MODEL_DIR / "als_model.npz",
)
print("Сохранено:", als_model_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/models/als_model.npz


### Сохранение ALS mappings

**Что делаем:** записываем исходные user/item ID  
**Зачем:** factor indices без mappings не имеют смысла  
**Что получим:** два JSON-файла

In [57]:
save_json(
    final_interactions.index_to_user,
    MODEL_DIR / "mappings" / "als_user_ids.json",
)
save_json(
    final_interactions.index_to_item,
    MODEL_DIR / "mappings" / "als_article_ids.json",
)
print("ALS mappings сохранены")

ALS mappings сохранены


### ALS test report

**Что делаем:** добавляем model name и техническую статистику к test metrics  
**Зачем:** notebook 10 загрузит один компактный CSV  
**Что получим:** `als_metrics.csv`

In [58]:
als_test_report = {
    "model": "ALS",
    **test_metrics,
    "users_evaluated": len(test_ground_truth_sample),
    "average_candidates": test_candidates.groupby("customer_id").size().mean(),
    "training_time": test_training_time,
    "notes": f"ALS Top-{CANDIDATE_LIMIT} candidates",
}
als_metrics_path = REPORT_DIR / "als_metrics.csv"
pd.DataFrame([als_test_report]).to_csv(als_metrics_path, index=False)
display(pd.Series(als_test_report))

,0
model,ALS
Candidate Recall,0.04085
Recall@12,0.004767
MAP@12,0.001333
HitRate@12,0.0055
users_evaluated,2000
average_candidates,200.0
training_time,53.61112
notes,ALS Top-200 candidates
